<a href="https://colab.research.google.com/github/sunidhi1703/Dream11/blob/main/interIIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install pulp

# Import all libraries
import pandas as pd
import numpy as np
import json
import os
import re
import zipfile
import pickle
import warnings
from glob import glob
from collections import defaultdict
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpStatus

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 65.2 MB/s eta 0:00:00


In [4]:
# Fantasy Point Calculation Engine
def calculate_batting_points(stats):
    points = 0
    runs = stats.get('runs', 0)
    balls_faced = stats.get('balls_faced', 0)
    fours = stats.get('fours', 0)
    sixes = stats.get('sixes', 0)
    role = stats.get('role', 'BAT')
    is_out = stats.get('is_out', False)
    points += runs + fours * 1 + sixes * 2
    if runs == 0 and is_out and role in ['WK', 'BAT', 'AR']:
        points -= 2
    if balls_faced >= 10:
        sr = (runs / balls_faced) * 100
        if sr >= 170: points += 6
        elif 150 <= sr < 170: points += 4
        elif 130 <= sr < 150: points += 2
        elif 50 <= sr < 70: points -= 2
        elif sr < 50: points -= 4
    return points

def calculate_bowling_points(stats):
    points = 0
    wickets = stats.get('wickets', 0)
    lbw_bowled = stats.get('lbw_bowled', 0)
    overs = stats.get('overs', 0)
    runs_conceded = stats.get('runs_conceded', 0)
    maidens = stats.get('maidens', 0)
    points += wickets * 25 + lbw_bowled * 8
    if wickets >= 5: points += 16
    elif wickets == 4: points += 10
    elif wickets == 3: points += 6
    points += maidens * 12
    if overs >= 2:
        er = runs_conceded / overs if overs > 0 else 0
        if er <= 5.0: points += 6
        elif 5.01 <= er <= 6.5: points += 4
        elif 6.51 <= er <= 8.0: points += 2
        elif 10.0 <= er <= 11.0: points -= 2
        elif er > 11.0: points -= 4
    return points

def calculate_fielding_points(stats):
    points = 0
    catches = stats.get('catches', 0)
    stumpings = stats.get('stumpings', 0)
    run_outs_direct = stats.get('run_outs_direct', 0)
    run_outs_shared = stats.get('run_outs_shared', 0)
    points += catches * 8
    if catches >= 3: points += 4
    points += stumpings * 12 + run_outs_direct * 12 + run_outs_shared * 6
    return points

def get_total_fantasy_points(player_match_stats):
    bat_pts = calculate_batting_points(player_match_stats)
    bowl_pts = calculate_bowling_points(player_match_stats)
    field_pts = calculate_fielding_points(player_match_stats)
    return bat_pts + bowl_pts + field_pts

# Role Handling Engine
try:
    df_roles_season = pd.read_csv('player_roles_by_season.csv')
    df_roles_global = pd.read_csv('player_roles_global.csv')
    print("Role CSVs loaded successfully.")
except FileNotFoundError:
    print("WARNING: Role CSV files not found. Make sure they are uploaded.")
    df_roles_season = pd.DataFrame(columns=['player_id', 'season', 'role'])
    df_roles_global = pd.DataFrame(columns=['player_id', 'role'])

def get_player_role(player_id, season, df_season, df_global):
    seasonal_role = df_season[
        (df_season['player_id'] == player_id) & (df_season['season'] == season)
    ]
    if not seasonal_role.empty:
        return seasonal_role.iloc[0]['role']
    global_role = df_global[df_global['player_id'] == player_id]
    if not global_role.empty:
        return global_role.iloc[0]['role']
    return 'BAT' # Default for new player

In [5]:
def parse_match_json(filepath, player_registry):
    """
    Parses a single JSON file to extract fantasy stats of each player
    Makes a list of dictionaries, one for each player in the match.
    """
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return []

    #  Get Match Info
    match_id = re.sub(r'\.json$', '', os.path.basename(filepath))
    match_date = data['info']['dates'][0]
    venue = data['info'].get('venue', 'Unknown Venue')
    season = int(match_date.split('-')[0])
    teams = data['info']['teams']

    #  Build Player ID Mappings
    name_to_id = {name: str(pid) for name, pid in data['info']['registry']['people'].items()}
    id_to_team = {}
    for team in teams:
        for player_name in data['info']['players'].get(team, []):
            if player_name in name_to_id:
                player_id = name_to_id[player_name]
                id_to_team[player_id] = team
                if player_id not in player_registry:
                    player_registry[player_id] = player_name

    #  Initialize Stats Dictionaries
    batting_stats = defaultdict(lambda: defaultdict(int))
    bowling_stats = defaultdict(lambda: defaultdict(int))
    fielding_stats = defaultdict(lambda: defaultdict(int))
    balls_faced = defaultdict(int)
    runs_conceded = defaultdict(int)
    legal_balls_bowled = defaultdict(int)
    runs_in_over = defaultdict(lambda: defaultdict(int))
    wickets_in_match = defaultdict(int)
    out_status = defaultdict(bool)
    final_stats_list = [] # Initialize the list here

    #  Loop Through All Deliveries
    for inning in data['innings']:
        for over in inning['overs']:
            over_num = over['over']
            for ball in over['deliveries']:
                batter_name = ball['batter']
                bowler_name = ball['bowler']
                batter_id = name_to_id.get(batter_name)
                bowler_id = name_to_id.get(bowler_name)
                if not batter_id or not bowler_id: continue

                runs = ball['runs']['batter']
                batting_stats[batter_id]['runs'] += runs
                if runs == 4: batting_stats[batter_id]['fours'] += 1
                if runs == 6: batting_stats[batter_id]['sixes'] += 1

                is_legal_delivery = True
                if 'wides' in ball.get('extras', {}):
                    runs_conceded[bowler_id] += ball['extras']['wides']
                    is_legal_delivery = False
                if 'noballs' in ball.get('extras', {}):
                    runs_conceded[bowler_id] += ball['extras']['noballs']

                if is_legal_delivery:
                    legal_balls_bowled[bowler_id] += 1
                    balls_faced[batter_id] += 1

                runs_conceded[bowler_id] += runs
                runs_in_over[bowler_id][over_num] += ball['runs']['total']

                if 'wickets' in ball:
                    for wicket in ball['wickets']:
                        kind = wicket['kind']
                        player_out_id = name_to_id.get(wicket['player_out'])
                        if player_out_id: out_status[player_out_id] = True

                        if kind == 'run out':
                            fielders = wicket.get('fielders', [])
                            if len(fielders) == 1:
                                f_id = name_to_id.get(fielders[0]['name'])
                                if f_id: fielding_stats[f_id]['run_outs_direct'] += 1
                            elif len(fielders) >= 2:
                                for f in fielders:
                                    f_id = name_to_id.get(f['name'])
                                    if f_id: fielding_stats[f_id]['run_outs_shared'] += 1
                        else:
                            bowling_stats[bowler_id]['wickets'] += 1
                            wickets_in_match[bowler_id] += 1
                            if kind in ['bowled', 'lbw']:
                                bowling_stats[bowler_id]['lbw_bowled'] += 1
                            if kind == 'caught':
                                f_id = name_to_id.get(wicket['fielders'][0]['name'])
                                if f_id: fielding_stats[f_id]['catches'] += 1
                            if kind == 'stumped':
                                f_id = name_to_id.get(wicket['fielders'][0]['name'])
                                if f_id: fielding_stats[f_id]['stumpings'] += 1

    all_player_ids = set(id_to_team.keys())
    for pid in all_player_ids:
        b_stats = bowling_stats[pid]
        b_stats['runs_conceded'] = runs_conceded[pid]
        b_stats['overs'] = legal_balls_bowled[pid] / 6.0
        for over_runs in runs_in_over[pid].values():
            if over_runs == 0: b_stats['maidens'] += 1
        b_stats['3_wickets'] = 1 if wickets_in_match[pid] == 3 else 0
        b_stats['4_wickets'] = 1 if wickets_in_match[pid] == 4 else 0
        b_stats['5_wickets'] = 1 if wickets_in_match[pid] >= 5 else 0

        f_stats = fielding_stats[pid]
        bat_stats = batting_stats[pid]
        bat_stats['balls_faced'] = balls_faced[pid]
        bat_stats['is_out'] = out_status[pid]

        role = get_player_role(pid, season, df_roles_season, df_roles_global)
        bat_stats['role'] = role

        combined_stats = {**bat_stats, **b_stats, **f_stats}
        total_fp = get_total_fantasy_points(combined_stats)

        final_stats_list.append({
            'match_id': match_id,
            'player_id': pid,
            'player_name': player_registry.get(pid, 'Unknown'),
            'team': id_to_team[pid],
            'match_date': match_date,
            'venue': venue,
            'season': season,
            'role': role,
            'fantasy_points': total_fp,
            **combined_stats
        })
    return final_stats_list


In [8]:
# Unzip your 'training_data.zip' file
zip_file_path = 'training_data.zip'

if os.path.exists(zip_file_path):
    print(f"Found '{zip_file_path}'. Unzipping...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall('.') # Extracts to the current directory
    print("Unzipping complete. 'Input_Matches_DATA' folder should now be available.")
else:
    print(f"WARNING: '{zip_file_path}' not found. Please upload it.")

all_match_logs = []
player_registry = {}

# Look inside the folder that was in your zip file.
json_files = glob('Input_Matches_DATA/*.json')

print(f"\nFound {len(json_files)} match JSON files to process...")

if not json_files:
    print(" WARNING: No JSON files found inside the 'Input_Matches_DATA' folder.")
else:
    for f in json_files:
        match_logs = parse_match_json(f, player_registry)
        all_match_logs.extend(match_logs)

    # Create the Master DataFrame
    df_logs = pd.DataFrame(all_match_logs)

    if df_logs.empty:
        print("ERROR: The final DataFrame is empty. Check your JSON files and parser logic.")
    else:
        df_logs['match_date'] = pd.to_datetime(df_logs['match_date'])
        df_logs = df_logs.sort_values(by=['match_date', 'match_id', 'player_id']).reset_index(drop=True)

        print("\nProcessing Complete!")
        print(f"Created master log DataFrame with {len(df_logs)} player-match records.")

        # Save
        df_logs.to_csv('player_match_logs.csv', index=False)
        print("\nSaved all data to 'player_match_logs.csv'")


Found 0 match JSON files to process...


In [9]:
# Load the dataset
try:
    df_logs = pd.read_csv('player_match_logs.csv')
    df_logs['match_date'] = pd.to_datetime(df_logs['match_date'])
    print("Loaded 'player_match_logs.csv' successfully.")
except FileNotFoundError:
    print("🚨 Error: 'player_match_logs.csv' not found. Please re-run Cell 4.")
    raise FileNotFoundError("Missing player_match_logs.csv")

# Add Opponent Features
print("Adding Opponent data...")
teams_per_match = df_logs.groupby('match_id')['team'].unique().to_frame()
teams_per_match['team_1'] = teams_per_match['team'].apply(lambda x: x[0] if len(x) > 0 else None)
teams_per_match['team_2'] = teams_per_match['team'].apply(lambda x: x[1] if len(x) > 1 else None)
df_logs = df_logs.merge(teams_per_match.drop(columns=['team']), on='match_id', how='left')
df_logs['opponent'] = np.where(df_logs['team'] == df_logs['team_1'], df_logs['team_2'], df_logs['team_1'])
original_rows = len(df_logs)
df_logs = df_logs.dropna(subset=['opponent'])
print(f"Dropped {original_rows - len(df_logs)} rows with missing opponent data.")

# Sort data
df_logs = df_logs.sort_values(by=['player_id', 'match_date'])

print("Creating improved historical (lagged) features...")
gb = df_logs.groupby('player_id')

# FP Features
df_logs['fp_last_1'] = gb['fantasy_points'].shift(1)
df_logs['fp_rolling_5'] = gb['fantasy_points'].shift(1).rolling(5, min_periods=1).mean()
df_logs['fp_rolling_10'] = gb['fantasy_points'].shift(1).rolling(10, min_periods=1).mean()
df_logs['fp_std_10'] = gb['fantasy_points'].shift(1).rolling(10, min_periods=1).std()
df_logs['fp_ewm_5'] = gb['fantasy_points'].shift(1).ewm(span=5, adjust=False).mean()

# Batting/Bowling Features
df_logs['runs_rolling_5'] = gb['runs'].shift(1).rolling(5, min_periods=1).mean()
df_logs['balls_faced_rolling_5'] = gb['balls_faced'].shift(1).rolling(5, min_periods=1).mean()
df_logs['wickets_rolling_5'] = gb['wickets'].shift(1).rolling(5, min_periods=1).mean()
df_logs['overs_rolling_5'] = gb['overs'].shift(1).rolling(5, min_periods=1).mean()
df_logs['runs_conceded_rolling_5'] = gb['runs_conceded'].shift(1).rolling(5, min_periods=1).mean()

# Opponent-Specific Features
print("Creating opponent-specific features...")
fp_vs_opp_expanding = df_logs.groupby(['player_id', 'opponent'])['fantasy_points'].expanding().mean()
fp_vs_opp_shifted = fp_vs_opp_expanding.groupby(level=[0, 1]).shift(1)
df_logs['fp_vs_opponent_mean'] = fp_vs_opp_shifted.reset_index(level=[0, 1], drop=True)

print("Creating venue-specific features...")
fp_vs_venue_expanding = df_logs.groupby(['player_id', 'venue'])['fantasy_points'].expanding().mean()
fp_vs_venue_shifted = fp_vs_venue_expanding.groupby(level=[0, 1]).shift(1)
df_logs['fp_vs_venue_mean'] = fp_vs_venue_shifted.reset_index(level=[0, 1], drop=True)

# Fallback: If no history at venue, use recent 5-game avg
df_logs['fp_vs_venue_mean'] = df_logs['fp_vs_venue_mean'].fillna(df_logs['fp_rolling_5'])
df_logs['fp_vs_venue_mean'] = df_logs['fp_vs_venue_mean'].fillna(0)

# Fill NaNs
df_logs['fp_std_10'] = df_logs['fp_std_10'].fillna(0)
df_logs['fp_vs_opponent_mean'] = df_logs['fp_vs_opponent_mean'].fillna(df_logs['fp_rolling_5'])
df_logs['fp_vs_opponent_mean'] = df_logs['fp_vs_opponent_mean'].fillna(0)

# Handle Categorical Features/
print("Encoding categorical 'role' and 'opponent' features...")
df_model_data = pd.get_dummies(df_logs, columns=['role', 'opponent'], drop_first=False)

df_model_data = df_model_data.dropna(subset=['fp_last_1'])
print(f"\nCreated model dataset with {len(df_model_data)} records.")

TARGET = 'fantasy_points'
role_cols = [col for col in df_model_data.columns if col.startswith('role_')]
opponent_cols = [col for col in df_model_data.columns if col.startswith('opponent_')]
venue_cols = [col for col in df_model_data.columns if col.startswith('venue_')]

FEATURES = [
    'fp_last_1', 'fp_rolling_5', 'fp_rolling_10', 'fp_std_10', 'fp_ewm_5',
    'runs_rolling_5', 'balls_faced_rolling_5', 'wickets_rolling_5',
    'overs_rolling_5', 'runs_conceded_rolling_5', 'fp_vs_opponent_mean' 'fp_vs_opponent_mean',
    'fp_vs_venue_mean'
] + role_cols + opponent_cols

columns_to_save = [TARGET] + FEATURES + ['match_id', 'player_id', 'match_date']
columns_to_save = [col for col in columns_to_save if col in df_model_data.columns]
df_model_data[columns_to_save].to_csv('model_training_data.csv', index=False)

print("\n--- Phase 3 (Improved) Complete ---")
print(f"Saved new feature data to 'model_training_data.csv'. Total features: {len(FEATURES)}")

🚨 Error: 'player_match_logs.csv' not found. Please re-run Cell 4.


FileNotFoundError: Missing player_match_logs.csv

In [65]:
print("--- Starting Credits Engine (Phase 2) ---")

# Load Both Required Files
try:
    df_features = pd.read_csv('model_training_data.csv')
    df_features['match_date'] = pd.to_datetime(df_features['match_date'])
    df_logs = pd.read_csv('player_match_logs.csv')
    print("Loaded 'model_training_data.csv' and 'player_match_logs.csv'")
except FileNotFoundError:
    print("🚨 Error: CSV files not found. Please re-run Cell 4 and Cell 5.")
    raise FileNotFoundError("Missing required CSV files")

# Calculate Composite Score
df_features['composite_score'] = 0.7 * df_features['fp_rolling_10'] + 0.3 * (df_features['fp_rolling_10'] - df_features['fp_std_10'])


df_features = df_features.merge(
    df_logs[['match_id', 'player_id', 'role']].drop_duplicates(),
    on=['match_id', 'player_id'],
    how='left'
)
df_features = df_features.dropna(subset=['role'])

# Calculate Percentile Ranks
df_features['percentile_rank'] = df_features.groupby('role')['composite_score'].rank(pct=True)

bands = {
    'top':    {'pct': 0.90, 'min_cred': 10.5, 'max_cred': 11.0},
    'next':   {'pct': 0.70, 'min_cred': 9.0,  'max_cred': 10.0},
    'middle': {'pct': 0.30, 'min_cred': 7.0,  'max_cred': 8.5},
    'bottom': {'pct': 0.0,  'min_cred': 4.0,  'max_cred': 6.5}
}

def apply_credits(row):
    pct = row['percentile_rank']
    if pd.isna(pct): return 8.0

    if pct >= bands['top']['pct']:
        band = bands['top']
        band_pct_range = (1.0 - band['pct'])
        pos_in_band = (pct - band['pct']) / band_pct_range if band_pct_range > 0 else 0
    elif pct >= bands['next']['pct']:
        band = bands['next']
        band_pct_range = (bands['top']['pct'] - band['pct'])
        pos_in_band = (pct - band['pct']) / band_pct_range if band_pct_range > 0 else 0
    elif pct >= bands['middle']['pct']:
        band = bands['middle']
        band_pct_range = (bands['next']['pct'] - band['pct'])
        pos_in_band = (pct - band['pct']) / band_pct_range if band_pct_range > 0 else 0
    else:
        band = bands['bottom']
        band_pct_range = (bands['middle']['pct'] - band['pct'])
        pos_in_band = (pct - band['pct']) / band_pct_range if band_pct_range > 0 else 0

    credits = band['min_cred'] + pos_in_band * (band['max_cred'] - band['min_cred'])
    return credits

print("Calculating percentile-based credits for all players...")
df_features['credits'] = df_features.apply(apply_credits, axis=1)

# -Newcomer
df_logs_sorted = df_logs.sort_values(by=['player_id', 'match_date'])
df_logs_sorted['appearance_count'] = df_logs_sorted.groupby('player_id').cumcount() + 1
df_features = df_features.merge(
    df_logs_sorted[['match_id', 'player_id', 'appearance_count']],
    on=['match_id', 'player_id'],
    how='left'
)
df_features = df_features.dropna(subset=['appearance_count'])
df_features['appearance_count'] = df_features['appearance_count'].astype(int)

role_median_credits = df_features[df_features['appearance_count'] >= 10].groupby('role')['credits'].median().to_dict()
print(f"Role medians computed: {role_median_credits}")

def newcomer_clamp(row):
    if row['appearance_count'] < 10:
        median = role_median_credits.get(row['role'], 8.0)
        return np.clip(median, median - 0.5, median + 0.5)
    else:
        return row['credits']

print("Applying newcomer clamp...")
df_features['credits'] = df_features.apply(newcomer_clamp, axis=1)

# --- 7. Final Rounding and Saving ---
df_features['credits'] = round(np.clip(df_features['credits'], 4.0, 11.0), 2)
df_features.to_csv('model_training_data_with_credits.csv', index=False)

print("\n Credits Engine Complete")
print("Saved new data (with all features + credits) to 'model_training_data_with_credits.csv'")

--- Starting Credits Engine (Phase 2) ---
Loaded 'model_training_data.csv' and 'player_match_logs.csv'
Calculating percentile-based credits for all players...
Role medians computed: {'AR': 7.961575875486382, 'BAT': 7.896912108234378, 'BOWL': 7.89885445342332, 'WK': 7.837938005390836}
Applying newcomer clamp...

--- Credits Engine Complete ---
Saved new data (with all features + credits) to 'model_training_data_with_credits.csv'


In [66]:
# Load the Feature-Engineered Data
try:
    df_model_data = pd.read_csv('model_training_data_with_credits.csv')
    df_model_data['match_date'] = pd.to_datetime(df_model_data['match_date'])
    print("Loaded 'model_training_data_with_credits.csv' successfully.")
except FileNotFoundError:
    print("🚨 Error: 'model_training_data_with_credits.csv' not found.")
    print("Please re-run Cell 5 and Cell 6 successfully.")
    raise FileNotFoundError

# Define Features (X) and Target (y)
TARGET = 'fantasy_points'
COLS_TO_EXCLUDE = [TARGET, 'match_id', 'player_id', 'match_date', 'role', 'appearance_count', 'composite_score', 'percentile_rank']
FEATURES = [col for col in df_model_data.columns if col not in COLS_TO_EXCLUDE]
print(f"--- Training model with {len(FEATURES)} features ---")

X = df_model_data[FEATURES]
y = df_model_data[TARGET]

print("\nCreating time-aware validation split...")
split_date = df_model_data['match_date'].quantile(0.8, interpolation='nearest')
train_idx = df_model_data['match_date'] < split_date
val_idx = df_model_data['match_date'] >= split_date

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
print(f"Training on {len(X_train)} records (matches before {split_date.date()})")
print(f"Validating on {len(X_val)} records (matches on/after {split_date.date()})")

# Initialize and Train XGBoost Model
print("\n--- Training XGBoost Model ---")
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=4,
    subsample=0.8,
    early_stopping_rounds=100,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

#Evaluate the Model
print("\n--- Model Evaluation ---")
preds = model.predict(X_val)
mae = mean_absolute_error(y_val, preds)
print(f"Validation Mean Absolute Error (MAE): {mae:.4f}")

# Save the Model Artifact
print("\n--- Saving Model Artifact ---")
os.makedirs('model_artifacts', exist_ok=True)
model_filename = 'model_artifacts/ProductUI_Model.json'
model.save_model(model_filename)
print(f"Model saved successfully to '{model_filename}'")

Loaded 'model_training_data_with_credits.csv' successfully.
--- Training model with 35 features ---

Creating time-aware validation split...
Training on 19939 records (matches before 2023-04-04)
Validating on 4993 records (matches on/after 2023-04-04)

--- Training XGBoost Model ---
[0]	validation_0-rmse:32.50193
[100]	validation_0-rmse:31.94389
[200]	validation_0-rmse:31.95238
[216]	validation_0-rmse:31.95586

--- Model Evaluation ---
Validation Mean Absolute Error (MAE): 25.1443

--- Saving Model Artifact ---
✅ Model saved successfully to 'model_artifacts/ProductUI_Model.json'


In [73]:
import pandas as pd
import numpy as np
import json
import os
import xgboost as xgb
import warnings
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpStatus

warnings.filterwarnings('ignore')

try:
    df_roles_season = pd.read_csv('player_roles_by_season.csv')
    df_roles_global = pd.read_csv('player_roles_global.csv')
    print("Role CSVs loaded successfully.")
except FileNotFoundError:
    print("🚨 FATAL: Role CSV files not found. Upload them and re-run Cell 2.")
    raise

def get_player_role(player_id, season, df_season, df_global):
    seasonal_role = df_season[
        (df_season['player_id'] == player_id) & (df_season['season'] == season)
    ]
    if not seasonal_role.empty:
        return seasonal_role.iloc[0]['role']
    global_role = df_global[df_global['player_id'] == player_id]
    if not global_role.empty:
        return global_role.iloc[0]['role']
    return 'BAT'

def calculate_batting_points(stats):
    points = 0
    runs = stats.get('runs', 0)
    balls_faced = stats.get('balls_faced', 0)
    fours = stats.get('fours', 0)
    sixes = stats.get('sixes', 0)
    role = stats.get('role', 'BAT')
    is_out = stats.get('is_out', False)
    points += runs + fours * 1 + sixes * 2
    if runs == 0 and is_out and role in ['WK', 'BAT', 'AR']: points -= 2
    if balls_faced >= 10:
        sr = (runs / balls_faced) * 100
        if sr >= 170: points += 6
        elif 150 <= sr < 170: points += 4
        elif 130 <= sr < 150: points += 2
        elif 50 <= sr < 70: points -= 2
        elif sr < 50: points -= 4
    return points

def calculate_bowling_points(stats):
    points = 0
    wickets = stats.get('wickets', 0)
    lbw_bowled = stats.get('lbw_bowled', 0)
    overs = stats.get('overs', 0)
    runs_conceded = stats.get('runs_conceded', 0)
    maidens = stats.get('maidens', 0)
    points += wickets * 25 + lbw_bowled * 8
    if wickets >= 5: points += 16
    elif wickets == 4: points += 10
    elif wickets == 3: points += 6
    points += maidens * 12
    if overs >= 2:
        er = runs_conceded / overs if overs > 0 else 0
        if er <= 5.0: points += 6
        elif 5.01 <= er <= 6.5: points += 4
        elif 6.51 <= er <= 8.0: points += 2
        elif 10.0 <= er <= 11.0: points -= 2
        elif er > 11.0: points -= 4
    return points

def calculate_fielding_points(stats):
    points = 0
    catches = stats.get('catches', 0)
    stumpings = stats.get('stumpings', 0)
    run_outs_direct = stats.get('run_outs_direct', 0)
    run_outs_shared = stats.get('run_outs_shared', 0)
    points += catches * 8
    if catches >= 3: points += 4
    points += stumpings * 12 + run_outs_direct * 12 + run_outs_shared * 6
    return points

def get_total_fantasy_points(player_match_stats):
    bat_pts = calculate_batting_points(player_match_stats)
    bowl_pts = calculate_bowling_points(player_match_stats)
    field_pts = calculate_fielding_points(player_match_stats)
    return bat_pts + bowl_pts + field_pts

def parse_match_json(filepath, player_registry):
    try:
        with open(filepath, 'r') as f: data = json.load(f)
    except Exception as e:
        print(f"Error reading {filepath}: {e}"); return []
    match_id = re.sub(r'\.json$', '', os.path.basename(filepath))
    match_date = data['info']['dates'][0]
    season = int(match_date.split('-')[0])
    venue = data['info'].get('venue', 'Unknown Venue') # <-- Includes Venue
    teams = data['info']['teams']
    name_to_id = {name: str(pid) for name, pid in data['info']['registry']['people'].items()}
    id_to_team = {}
    for team in teams:
        for player_name in data['info']['players'].get(team, []):
            if player_name in name_to_id:
                player_id = name_to_id[player_name]
                id_to_team[player_id] = team
                if player_id not in player_registry:
                    player_registry[player_id] = player_name
    batting_stats = defaultdict(lambda: defaultdict(int))
    bowling_stats = defaultdict(lambda: defaultdict(int))
    fielding_stats = defaultdict(lambda: defaultdict(int))
    balls_faced = defaultdict(int)
    runs_conceded = defaultdict(int)
    legal_balls_bowled = defaultdict(int)
    runs_in_over = defaultdict(lambda: defaultdict(int))
    wickets_in_match = defaultdict(int)
    out_status = defaultdict(bool)
    final_stats_list = []
    for inning in data['innings']:
        for over in inning['overs']:
            over_num = over['over']
            for ball in over['deliveries']:
                batter_name = ball['batter']
                bowler_name = ball['bowler']
                batter_id = name_to_id.get(batter_name)
                bowler_id = name_to_id.get(bowler_name)
                if not batter_id or not bowler_id: continue
                runs = ball['runs']['batter']
                batting_stats[batter_id]['runs'] += runs
                if runs == 4: batting_stats[batter_id]['fours'] += 1
                if runs == 6: batting_stats[batter_id]['sixes'] += 1
                is_legal_delivery = True
                if 'wides' in ball.get('extras', {}):
                    runs_conceded[bowler_id] += ball['extras']['wides']
                    is_legal_delivery = False
                if 'noballs' in ball.get('extras', {}):
                    runs_conceded[bowler_id] += ball['extras']['noballs']
                if is_legal_delivery:
                    legal_balls_bowled[bowler_id] += 1
                    balls_faced[batter_id] += 1
                runs_conceded[bowler_id] += runs
                runs_in_over[bowler_id][over_num] += ball['runs']['total']
                if 'wickets' in ball:
                    for wicket in ball['wickets']:
                        kind = wicket['kind']
                        player_out_id = name_to_id.get(wicket['player_out'])
                        if player_out_id: out_status[player_out_id] = True
                        if kind == 'run out':
                            fielders = wicket.get('fielders', [])
                            if len(fielders) == 1:
                                f_id = name_to_id.get(fielders[0]['name'])
                                if f_id: fielding_stats[f_id]['run_outs_direct'] += 1
                            elif len(fielders) >= 2:
                                for f in fielders:
                                    f_id = name_to_id.get(f['name'])
                                    if f_id: fielding_stats[f_id]['run_outs_shared'] += 1
                        else:
                            bowling_stats[bowler_id]['wickets'] += 1
                            wickets_in_match[bowler_id] += 1
                            if kind in ['bowled', 'lbw']: bowling_stats[bowler_id]['lbw_bowled'] += 1
                            if kind == 'caught':
                                f_id = name_to_id.get(wicket['fielders'][0]['name'])
                                if f_id: fielding_stats[f_id]['catches'] += 1
                            if kind == 'stumped':
                                f_id = name_to_id.get(wicket['fielders'][0]['name'])
                                if f_id: fielding_stats[f_id]['stumpings'] += 1
    all_player_ids = set(id_to_team.keys())
    for pid in all_player_ids:
        b_stats = bowling_stats[pid]
        b_stats['runs_conceded'] = runs_conceded[pid]
        b_stats['overs'] = legal_balls_bowled[pid] / 6.0
        for over_runs in runs_in_over[pid].values():
            if over_runs == 0: b_stats['maidens'] += 1
        b_stats['3_wickets'] = 1 if wickets_in_match[pid] == 3 else 0
        b_stats['4_wickets'] = 1 if wickets_in_match[pid] == 4 else 0
        b_stats['5_wickets'] = 1 if wickets_in_match[pid] >= 5 else 0
        f_stats = fielding_stats[pid]
        bat_stats = batting_stats[pid]
        bat_stats['balls_faced'] = balls_faced[pid]
        bat_stats['is_out'] = out_status[pid]
        role = get_player_role(pid, season, df_roles_season, df_roles_global)
        bat_stats['role'] = role
        combined_stats = {**bat_stats, **b_stats, **f_stats}
        total_fp = get_total_fantasy_points(combined_stats)
        final_stats_list.append({
            'match_id': match_id, 'player_id': pid,
            'player_name': player_registry.get(pid, 'Unknown'),
            'team': id_to_team[pid], 'match_date': match_date,
            'season': season, 'role': role, 'venue': venue,
            'fantasy_points': total_fp, **combined_stats
        })
    return final_stats_list


def select_optimal_team(players_df, budget=100.0):
    player_ids = players_df['player_id'].tolist()
    pred_fp = pd.Series(players_df.predicted_fp.values, index=player_ids).to_dict()
    credits = pd.Series(players_df.credits.values, index=player_ids).to_dict()
    roles = pd.Series(players_df.role.values, index=player_ids).to_dict()
    teams = pd.Series(players_df.team.values, index=player_ids).to_dict()
    team_names = players_df['team'].unique()
    prob = LpProblem("Dream11_Team_Selection", LpMaximize)
    player_vars = LpVariable.dicts("Player", player_ids, 0, 1, 'Binary')
    prob += lpSum([pred_fp[p] * player_vars[p] for p in player_ids]), "Total_Predicted_FP"
    prob += lpSum([player_vars[p] for p in player_ids]) == 11, "Total_Players"
    prob += lpSum([credits[p] * player_vars[p] for p in player_ids]) <= budget, "Total_Credits"
    prob += lpSum([player_vars[p] for p in player_ids if roles[p] == 'WK']) >= 1, "Min_WK"
    prob += lpSum([player_vars[p] for p in player_ids if roles[p] == 'WK']) <= 4, "Max_WK"
    prob += lpSum([player_vars[p] for p in player_ids if roles[p] == 'BAT']) >= 3, "Min_BAT"
    prob += lpSum([player_vars[p] for p in player_ids if roles[p] == 'BAT']) <= 6, "Max_BAT"
    prob += lpSum([player_vars[p] for p in player_ids if roles[p] == 'AR']) >= 1, "Min_AR"
    prob += lpSum([player_vars[p] for p in player_ids if roles[p] == 'AR']) <= 4, "Max_AR"
    prob += lpSum([player_vars[p] for p in player_ids if roles[p] == 'BOWL']) >= 3, "Min_BOWL"
    prob += lpSum([player_vars[p] for p in player_ids if roles[p] == 'BOWL']) <= 6, "Max_BOWL"
    for team in team_names:
        prob += lpSum([player_vars[p] for p in player_ids if teams[p] == team]) <= 7, f"Max_{team}"
        prob += lpSum([player_vars[p] for p in player_ids if teams[p] == team]) >= 1, f"Min_{team}"
    prob.solve()
    if LpStatus[prob.status] == 'Optimal':
        selected_ids = [p for p in player_ids if player_vars[p].varValue == 1]
        return players_df[players_df['player_id'].isin(selected_ids)]
    else:
        print(f"🚨 Error: No optimal solution found. Status: {LpStatus[prob.status]}"); return None

print("\n--- All Helper Functions Defined ---")


try:
    df_train_data = pd.read_csv('model_training_data_with_credits.csv')
    print("Loaded 'model_training_data_with_credits.csv' to get feature list.")
except FileNotFoundError:
    print("🚨 Error: 'model_training_data_with_credits.csv' not found.")
    print("Please re-run Cells 5, 6, and 7 successfully.")
    raise FileNotFoundError

TARGET = 'fantasy_points'
COLS_TO_EXCLUDE = [TARGET, 'match_id', 'player_id', 'match_date', 'role', 'appearance_count', 'composite_score', 'percentile_rank']
FEATURES = [col for col in df_train_data.columns if col not in COLS_TO_EXCLUDE]
print(f"--- Loaded {len(FEATURES)} features from training data ---")

TEST_MATCH_FILE = 'Input_Matches_DATA/336003.json'

try:
    model_filename = 'model_artifacts/ProductUI_Model.json'
    model = xgb.XGBRegressor()
    model.load_model(model_filename)
    print(f"Loaded '{model_filename}' successfully.")

    df_logs = pd.read_csv('player_match_logs.csv')
    df_logs['match_date'] = pd.to_datetime(df_logs['match_date'])
    teams_per_match = df_logs.groupby('match_id')['team'].unique().to_frame()
    teams_per_match['team_1'] = teams_per_match['team'].apply(lambda x: x[0] if len(x) > 0 else None)
    teams_per_match['team_2'] = teams_per_match['team'].apply(lambda x: x[1] if len(x) > 1 else None)
    df_logs = df_logs.merge(teams_per_match.drop(columns=['team']), on='match_id', how='left')
    df_logs['opponent'] = np.where(df_logs['team'] == df_logs['team_1'], df_logs['team_2'], df_logs['team_1'])
    print("Loaded and pre-processed 'player_match_logs.csv' (full history).")

except FileNotFoundError as e:
    print(f" Error: Missing a required file. {e}")
    raise e

try:
    df_logs_sorted = df_logs.sort_values(by=['player_id', 'match_date'])
    df_logs_sorted['appearance_count'] = df_logs_sorted.groupby('player_id').cumcount() + 1
    temp_credits_df = df_train_data[df_train_data['appearance_count'] >= 10]
    role_median_credits = temp_credits_df.groupby('role')['credits'].median().to_dict()
    print(f"Calculated fallback credit medians: {role_median_credits}")
except Exception as e:
    print(f"Warning: Could not calculate medians. Using hardcoded defaults. Error: {e}")
    role_median_credits = {'AR': 8.0, 'BAT': 8.5, 'BOWL': 8.0, 'WK': 8.5}



print(f"\n--- Starting Prediction Pipeline for {TEST_MATCH_FILE} ---")
test_players_features = []

try:
    with open(TEST_MATCH_FILE, 'r') as f:
        test_data = json.load(f)

    test_match_date = pd.to_datetime(test_data['info']['dates'][0])
    test_season = test_match_date.year
    teams = test_data['info']['teams']
    venue = test_data['info'].get('venue', 'Unknown Venue')
    team_1_name, team_2_name = teams[0], teams[1]

    name_to_id = {name: str(pid) for name, pid in test_data['info']['registry']['people'].items()}
    id_to_name = {str(pid): name for name, pid in test_data['info']['registry']['people'].items()}
    id_to_team = {}
    all_player_ids = []

    for team in teams:
        for player_name in test_data['info']['players'].get(team, []):
            if player_name in name_to_id:
                player_id = name_to_id[player_name]
                id_to_team[player_id] = team
                all_player_ids.append(player_id)
    print(f"Found {len(all_player_ids)} players in the squads for match on {test_match_date.date()}...")
except Exception as e:
    print(f"🚨 FATAL ERROR: Could not read or parse {TEST_MATCH_FILE}. Error: {e}")
    raise e

for pid in all_player_ids:

    player_history = df_logs[
        (df_logs['player_id'] == pid) &
        (df_logs['match_date'] < test_match_date)
    ].sort_values(by='match_date', ascending=False)

    role = get_player_role(pid, test_season, df_roles_season, df_roles_global)
    player_team = id_to_team.get(pid)
    opponent = team_2_name if player_team == team_1_name else team_1_name
    n_apps = len(player_history)

    if n_apps == 0:
        features = {
            'fp_last_1': 0, 'fp_rolling_5': 0, 'fp_rolling_10': 0, 'fp_std_10': 0,
            'fp_ewm_5': 0, 'runs_rolling_5': 0, 'balls_faced_rolling_5': 0,
            'wickets_rolling_5': 0, 'overs_rolling_5': 0, 'runs_conceded_rolling_5': 0,
            'fp_vs_opponent_mean': 0, 'fp_vs_venue_mean': 0
        }
    else:
        features = {
            'fp_last_1': player_history.iloc[0]['fantasy_points'],
            'fp_rolling_5': player_history.head(5)['fantasy_points'].mean(),
            'fp_rolling_10': player_history.head(10)['fantasy_points'].mean(),
            'fp_std_10': player_history.head(10)['fantasy_points'].std(),
            'fp_ewm_5': player_history['fantasy_points'].iloc[::-1].ewm(span=5, adjust=False).mean().iloc[-1],
            'runs_rolling_5': player_history.head(5)['runs'].mean(),
            'balls_faced_rolling_5': player_history.head(5)['balls_faced'].mean(),
            'wickets_rolling_5': player_history.head(5)['wickets'].mean(),
            'overs_rolling_5': player_history.head(5)['overs'].mean(),
            'runs_conceded_rolling_5': player_history.head(5)['runs_conceded'].mean()
        }
        history_vs_opponent = player_history[player_history['opponent'] == opponent]
        features['fp_vs_opponent_mean'] = history_vs_opponent['fantasy_points'].mean() if not history_vs_opponent.empty else features['fp_rolling_5']
        history_vs_venue = player_history[player_history['venue'] == venue]
        features['fp_vs_venue_mean'] = history_vs_venue['fantasy_points'].mean() if not history_vs_venue.empty else features['fp_rolling_5']

    features = {k: 0 if pd.isna(v) else v for k, v in features.items()}

    median = role_median_credits.get(role, 8.0)
    if n_apps < 10:
        credits = np.clip(median, median - 0.5, median + 0.5)
    else:
        mu = features['fp_rolling_10']
        std = features['fp_std_10']
        composite_score = 0.7 * mu + 0.3 * (mu - std)
        credits = np.interp(composite_score, [10, 60], [7.0, 10.5])

    credits = round(np.clip(credits, 4.0, 11.0), 2)

    test_players_features.append({
        'player_id': pid,
        'player_name': id_to_name.get(pid, 'Unknown'),
        'team': id_to_team.get(pid, 'Unknown'),
        'role': role,
        'opponent': opponent,
        'venue': venue,
        'credits': credits,
        **features
    })


df_test = pd.DataFrame(test_players_features)
df_test_encoded = pd.get_dummies(df_test, columns=['role', 'opponent', 'venue'])
X_test_aligned = pd.DataFrame(columns=FEATURES, index=df_test_encoded.index)
common_cols = [col for col in df_test_encoded.columns if col in FEATURES]
X_test_aligned[common_cols] = df_test_encoded[common_cols]
X_test_aligned = X_test_aligned.fillna(0)

df_test['predicted_fp'] = model.predict(X_test_aligned)

print("\n--- Running Constraints Solver to Select Optimal XI ---")
try:
    recommended_xi = select_optimal_team(df_test)
except NameError:
    print("🚨 FATAL ERROR: 'select_optimal_team' is not defined.")
    recommended_xi = None

if recommended_xi is not None:
    print("\n✅ --- YOUR RECOMMENDED DREAM11 TEAM --- ✅")
    print(recommended_xi[['player_name', 'team', 'role', 'credits', 'predicted_fp']].sort_values(by='predicted_fp', ascending=False).to_markdown(index=False))

    print("\n--- FINAL TEAM STATS ---")
    print(f"Total Predicted Points: {recommended_xi['predicted_fp'].sum():.2f}")
    print(f"Total Credits Used:     {recommended_xi['credits'].sum():.2f} / 100.0")
else:
    print("🚨 FAILED to find a feasible team. Check solver constraints and player data.")


if recommended_xi is not None:
    print("\n" + "="*60)
    print("--- STARTING EVALUATION: PREDICTED XI vs. DREAM XI ---")
    print("="*60 + "\n")

    player_registry = {} # Temporary registry
    actual_stats_list = parse_match_json(TEST_MATCH_FILE, player_registry)
    df_actual_stats = pd.DataFrame(actual_stats_list)
    df_actual_points = df_actual_stats[['player_id', 'fantasy_points']].rename(columns={'fantasy_points': 'actual_fp'})
    df_actual_points['actual_fp'] = df_actual_points['actual_fp'].fillna(0)

    # Create the "Ground Truth" DataFrame
    df_ground_truth = df_test.merge(df_actual_points, on='player_id', how='left')
    df_ground_truth['actual_fp'] = df_ground_truth['actual_fp'].fillna(0)

    # Find the "Dream XI"
    df_solver_dream_xi = df_ground_truth.copy()
    df_solver_dream_xi = df_solver_dream_xi.drop(columns=['predicted_fp'])
    df_solver_dream_xi = df_solver_dream_xi.rename(columns={'actual_fp': 'predicted_fp'})

    print("Running Solver to find the *Actual* Dream XI ")
    dream_xi = select_optimal_team(df_solver_dream_xi)

    # calculate and Show Comparison Metric
    if dream_xi is not None:
        print("\nACTUAL DREAM XI (Based on real performance) ")
        dream_xi_with_actual_points = dream_xi.merge(df_actual_points, on='player_id')
        print(dream_xi_with_actual_points[['player_name', 'team', 'role', 'credits', 'actual_fp']].sort_values(by='actual_fp', ascending=False).to_markdown(index=False))

        dream_xi_total_points = dream_xi_with_actual_points['actual_fp'].sum()
        print(f"\nDream XI Total Actual Points: {dream_xi_total_points:.2f}")
        print(f"Dream XI Total Credits Used:  {dream_xi['credits'].sum():.2f} / 100.0")

        print("\n\nYOUR PREDICTED XI's *Actual* Performance ")
        predicted_xi_players = recommended_xi[['player_id', 'player_name']]
        predicted_xi_actual_points = predicted_xi_players.merge(df_actual_points, on='player_id')

        predicted_xi_total_actual_points = predicted_xi_actual_points['actual_fp'].sum()
        print(f"Your Predicted XI's *Actual* Points: {predicted_xi_total_actual_points:.2f}")
        print(f"Your Predicted XI's Total Credits Used: {recommended_xi['credits'].sum():.2f} / 100.0")

        print("\n\nFINAL SCORE")
        ae_team_total = abs(dream_xi_total_points - predicted_xi_total_actual_points)
        print(f"Absolute Error (ae_team_total): {ae_team_total:.2f}")

    else:
        print(" FAILED to find the Dream XI. Solver error.")
else:
    print("\nSkipping evaluation because the Predicted XI could not be generated.")

Role CSVs loaded successfully.

--- All Helper Functions Defined ---
Loaded 'model_training_data_with_credits.csv' to get feature list.
--- Loaded 35 features from training data ---
Loaded 'model_artifacts/ProductUI_Model.json' successfully.
Loaded and pre-processed 'player_match_logs.csv' (full history).
Calculated fallback credit medians: {'AR': 7.96, 'BAT': 7.9, 'BOWL': 7.9, 'WK': 7.84}

--- Starting Prediction Pipeline for Input_Matches_DATA/336003.json ---
Found 22 players in the squads for match on 2008-05-03...

--- Running Constraints Solver to Select Optimal XI ---

✅ --- YOUR RECOMMENDED DREAM11 TEAM --- ✅
| player_name     | team                  | role   |   credits |   predicted_fp |
|:----------------|:----------------------|:-------|----------:|---------------:|
| SC Ganguly      | Kolkata Knight Riders | AR     |      7.96 |        44.7393 |
| PP Chawla       | Kings XI Punjab       | BOWL   |      7.9  |        27.1461 |
| JR Hopes        | Kings XI Punjab       | BOWL